# PIDNet-S seed1337 formal Val run
Five-channel, five-class paired replication. Test is locked.

In [ ]:
!pip install -q segmentation-models-pytorch==0.5.0 rasterio
from pathlib import Path
import subprocess
REPO=Path('/kaggle/working/lunar-linear')
if not (REPO/'.git').is_dir():
    subprocess.check_call(['git','clone','--branch','test-new-module','--single-branch','https://github.com/song110585-cpu/lunar-linear.git',str(REPO)])
else:
    subprocess.check_call(['git','pull','--ff-only','origin','test-new-module'],cwd=REPO)
PROJECT_DIR=REPO/'LTL-Net'
print(subprocess.check_output(['git','rev-parse','--short','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
import hashlib,json
CONFIG_PATH=PROJECT_DIR/'configs/v6_overlap40_pidnet_s_full_seed1337.json'
config=json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
data_paths=[p for p in Path('/kaggle/input').rglob('dataset_v6_random811_overlap40') if p.is_dir() and (p/'dataset_protocol.json').is_file()]
assert len(data_paths)==1,f'应唯一找到数据集，实际: {data_paths}'
DATA_DIR=data_paths[0]
for split,expected in config['expected_tiles'].items():
    images=list((DATA_DIR/split/'image').glob('*.tif'))+list((DATA_DIR/split/'image').glob('*.tiff'))
    masks=list((DATA_DIR/split/'mask').glob('*.tif'))+list((DATA_DIR/split/'mask').glob('*.tiff'))
    assert len(images)==len(masks)==expected,(split,len(images),len(masks),expected)
for filename,expected_hash in config['expected_metadata_sha256'].items():
    assert hashlib.sha256((DATA_DIR/filename).read_bytes()).hexdigest()==expected_hash,filename
weight_candidates=list(Path('/kaggle/input').rglob(config['pretrained_filename']))
weights=[p for p in weight_candidates if hashlib.md5(p.read_bytes()).hexdigest()==config['expected_pretrained_md5']]
assert weights,f'未找到MD5正确的 {config["pretrained_filename"]}: {weight_candidates}'
PRETRAINED=weights[0]
assert config['automatic_test_evaluation'] is False
print('数据:',DATA_DIR)
print('PIDNet-S ImageNet权重:',PRETRAINED)

In [ ]:
import sys,torch
sys.path.insert(0,str(PROJECT_DIR))
from models.pidnet_multiclass import PIDNetSmall,load_pidnet_imagenet_weights
smoke_model=PIDNetSmall().cuda().train()
loading=load_pidnet_imagenet_weights(smoke_model,PRETRAINED)
smoke_x=torch.randn(2,5,128,128,device='cuda')
with torch.amp.autocast('cuda'):
    smoke_y=smoke_model(smoke_x)
assert smoke_y.shape==(2,5,128,128) and torch.isfinite(smoke_y).all()
smoke_y.mean().backward()
assert loading['adapted_input_tensors']==1
assert loading['loaded_tensors']==config['expected_pretrained_loaded_tensors'],loading
print('smoke test passed:',loading)
del smoke_model,smoke_x,smoke_y
torch.cuda.empty_cache()

In [ ]:
OUTPUT_DIR=Path('/kaggle/working')
command=[sys.executable,str(PROJECT_DIR/'scripts/train_baseline.py'),'--model',config['model'],'--data-dir',str(DATA_DIR),'--output-dir',str(OUTPUT_DIR),'--pretrained-checkpoint',str(PRETRAINED),'--seed',str(config['seed']),'--epochs',str(config['epochs']),'--batch-size',str(config['batch_size']),'--accum-steps',str(config['accum_steps']),'--num-workers',str(config['num_workers']),'--run-name',config['run_name'],'--skip-test-evaluation']
print(' '.join(command),flush=True)
subprocess.check_call(command,cwd=PROJECT_DIR)

In [ ]:
RESULT_DIR=OUTPUT_DIR/f"result_{config['run_name']}"
metrics=json.loads((RESULT_DIR/'metrics.json').read_text(encoding='utf-8'))
assert metrics['model']=='PIDNet-S' and metrics['encoder']=='native_pidnet_s'
assert metrics['automatic_test_evaluation'] is False and metrics['test'] is None
assert metrics['pretrained_loading']['adapted_input_tensors']==1 and (RESULT_DIR/'best_model.pth').is_file()
print(json.dumps(metrics,ensure_ascii=False,indent=2))
print('请下载完整目录:',RESULT_DIR)